In [1]:
%matplotlib widget

import os
import matplotlib.pyplot as plt
import numpy as np
import os
from android_bot.android_capture import ScreenCapture

from android_bot import image_utils
from android_bot import android_actions
from blackjack.strategy import CardCounter, BasicStrategy, DeviatedBasicStrategy
import pandas as pd
from datetime import datetime

from android_bot import get_cards_tablet
import time

from android_bot.tablet_hilo_dev_player import Player


In [2]:
# Configure logging to see output from android_actions and other modules
import logging
from datetime import datetime
import os

# Create logs directory if it doesn't exist
os.makedirs('../logs', exist_ok=True)

# Create log filename with timestamp
log_filename = f"../logs/log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()  # This keeps console output as well
    ]
)

print(f"Logging to {log_filename}")

Logging to ../logs/log_20251225_212227.txt


In [3]:
logger = logging.getLogger(__name__)


In [4]:
def bet_size_ramp(tc_int):
    if tc_int >= 6:
        return 20
    elif tc_int >= 5:
        return 18
    elif tc_int >= 4:
        return 14
    elif tc_int >= 3:
        return 10
    elif tc_int >= 2:
        return 6
    elif tc_int >= 1:
        return 3
    else:
        return 1

In [5]:
def close_ads(actor, src_taker):
    ignore_region_xy = [[1700,1780], [1350, 1430]]
    def not_in_ignore_region(xy_position):
        return not (
            ignore_region_xy[0][0] < xy_position[0] < ignore_region_xy[0][1]
            and ignore_region_xy[1][0] < xy_position[1] < ignore_region_xy[1][1]
        )

    game_img = src_taker.get_screen()
    ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
    ads_crosses = list(filter(not_in_ignore_region, ads_crosses))

    while len(ads_crosses) > 0:
        for xy_position in ads_crosses:
            actor.click(*xy_position)
        time.sleep(1)
        game_img = src_taker.get_screen()
        ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
        ads_crosses = list(filter(not_in_ignore_region, ads_crosses))


def restart_table(actor, src_taker):
    # leave table
    actor.leave_table()

    game_img = src_taker.get_screen()
    leave_table_btns = get_cards_tablet.find_leave_table_button(game_img)
    while len(leave_table_btns) == 0:
        time.sleep(1)
        game_img = src_taker.get_screen()
        leave_table_btns = get_cards_tablet.find_leave_table_button(game_img)
    exit_btn = leave_table_btns[0]
    actor.click(*exit_btn)
    
    while True:
        close_ads(actor, src_taker)
        
        game_img = src_taker.get_screen()
        if get_cards_tablet.can_create_private_table(game_img):
            actor.create_private_table()
            
            time.sleep(2)
            game_img = src_taker.get_screen()
            if get_cards_tablet.can_create_private_table(game_img):
                # if still can create then try again
                continue
            else:
                break
        elif get_cards_tablet.is_table_empty(game_img):
            return

In [6]:
play_log_filename = f"../logs/play_{datetime.now().strftime("%Y_%m_%d_%H_%M_%S")}.csv"

# columns
# dealer_hand, player_hand_0, player_hand_1, total_bet, net_value, datetime up to second

def create_header_if_not_exists():
    if not os.path.exists(play_log_filename):
        with open(play_log_filename, 'w') as f:
            line = ",".join([
                "shoe_hand_idx",
                "dealer_hand",
                "player_hand_0",
                "player_hand_1",
                "total_bet",
                "net_value",
                "true_count",
                "remaining_cards",
                "datetime"
            ])
            f.write(line + "\n")

create_header_if_not_exists()

def hand_to_str_line(hand):
    return " ".join(str(rank) for rank in hand.cards)

def log_play_stats(shoe_hand_idx, round, counter):
    dealer_hand_str = hand_to_str_line(round.get_dealer_hand())
    player_hand_0_str = hand_to_str_line(round.player_hands[0])
    if len(round.player_hands) > 1:
        player_hand_1_str = hand_to_str_line(round.player_hands[1])
    else:
        player_hand_1_str = ""
    total_bet_str = str(round.get_player_bet())
    net_value_str = str(round.get_player_value())

    now = datetime.now()
    dt_string = now.strftime("%Y-%m-%d %H:%M:%S")
    tc_str = str(counter.get_true_count())
    rc_str = str(counter.remaining_cards)

    with open(play_log_filename, 'a') as f:
        line = ",".join([
            str(shoe_hand_idx),
            dealer_hand_str,
            player_hand_0_str,
            player_hand_1_str,
            total_bet_str,
            net_value_str,
            tc_str,
            rc_str,
            dt_string
        ])
        f.write(line)
        f.write("\n")


In [7]:
from android_bot.android_actions import AndroidBJTabletActor


src_taker = None
del src_taker

all_cards = []
net_value = []

src_taker = ScreenCapture()
game_img = src_taker.get_screen()
initial_penetration = 0 # get_cards_tablet.get_shoe_penetration(game_img)
logger.info(f"initial_penetration {initial_penetration}")
counter = CardCounter(n_decks=6, current_penetration=initial_penetration)
actor = AndroidBJTabletActor([250, 500, 1000, 2500, 5000])

2025-12-25 21:22:27,345 - __main__ - INFO - initial_penetration 0


In [8]:
# restart_table(actor, src_taker)

In [9]:
strategy = DeviatedBasicStrategy("../strategy", "../strategy")

In [10]:
shoe_hand_idx = 0

In [11]:
p = Player(actor, src_taker, counter, strategy)

In [12]:
p.reset_round()

In [13]:
while True:
    logger.info(f"start, true_count = {counter.get_true_count()}")
    int_tc = counter.get_integer_tc()
    bet = bet_size_ramp(int_tc) * 250

    p.play_round(bet)

    for hand in p.round.player_hands:
        all_cards.extend(hand.cards)
    all_cards.extend(p.round.dealer_hand.cards)
    net_value.append(p.round.get_player_value())
    
    log_play_stats(shoe_hand_idx, p.round, counter)
    
    shoe_hand_idx += 1
    
    true_count = counter.get_true_count() 
    if true_count < 0:
        logger.info(f"true count too low true_count {true_count}")
        restart_table(actor, src_taker)
        counter.reset()
        shoe_hand_idx = 0

    game_img = src_taker.get_screen()
    if get_cards_tablet.is_pre_shuffle(game_img):
        restart_table(actor, src_taker)
        counter.reset()
        shoe_hand_idx = 0

    time.sleep(1)

2025-12-25 21:22:29,273 - __main__ - INFO - start, true_count = 0.0
2025-12-25 21:22:29,300 - android_bot.android_actions - INFO - click at 1520.0, 1300.0
2025-12-25 21:22:29,823 - android_bot.android_actions - INFO - click at 1125.0, 800.0
2025-12-25 21:22:30,252 - android_bot.android_actions - INFO - click at 1300.0, 765.0
2025-12-25 21:22:31,173 - android_bot.android_actions - INFO - deal
2025-12-25 21:22:31,174 - android_bot.android_actions - INFO - click at 2140.0, 1300.0
2025-12-25 21:22:31,658 - android_bot.tablet_hilo_dev_player - INFO - wait_for_initial_cards
2025-12-25 21:22:37,572 - android_bot.tablet_hilo_dev_player - INFO - initial hand
2025-12-25 21:22:37,573 - android_bot.tablet_hilo_dev_player - INFO - Dealer A(1/11)
Player 10,10(20)[$250]
2025-12-25 21:22:37,573 - android_bot.tablet_hilo_dev_player - INFO - true count: -0.5048543689320388
2025-12-25 21:22:37,573 - android_bot.tablet_hilo_dev_player - INFO - wait_for_insurance_option
2025-12-25 21:22:38,299 - android_bo

KeyboardInterrupt: 

In [ ]:
print(p.round)

Last card: 7
Dealer 10,X (checked - no bj)
Player 2,4,10,7(23/bust)[$250]


In [ ]:
p.round.get_stage()

<BJStage.DEALER_CARD: 7>

In [ ]:
p.round.get_stage()

In [ ]:
close_ads(actor, src_taker)

In [ ]:
ignore_region_xy = [[1700,1780], [1350, 1430]]
def not_in_ignore_region(xy_position):
    return not (
        ignore_region_xy[0][0] < xy_position[0] < ignore_region_xy[0][1]
        and ignore_region_xy[1][0] < xy_position[1] < ignore_region_xy[1][1]
    )

game_img = src_taker.get_screen()
ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
ads_crosses = list(filter(not_in_ignore_region, ads_crosses))

while len(ads_crosses) > 0:
    for xy_position in ads_crosses:
        actor.click(*xy_position)
    time.sleep(2)
    game_img = src_taker.get_screen()
    ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
    ads_crosses = list(filter(not_in_ignore_region, ads_crosses))

In [ ]:
ads_crosses

In [ ]:
ignore_region_xy = [[1700,1780], [1350, 1430]]
def not_in_ignore_region(xy_position):
    return not (
        ignore_region_xy[0][0] < xy_position[0] < ignore_region_xy[0][1]
        and ignore_region_xy[1][0] < xy_position[1] < ignore_region_xy[1][1]
    )

game_img = src_taker.get_screen()
ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
ads_crosses = list(filter(not_in_ignore_region, ads_crosses))

In [ ]:
ads_crosses

[]